# 📊 Odin Daily Call Report Generator (Colab Edition) — **v2**

This interactive notebook wraps **`daily_odin_report.py`** and breaks it into easy‑to‑test units so you can:

* configure variables in one place  
* run each functional step independently  
* experiment with different report windows, batch sizes, etc.  

> **Important**: **Upload and import the script first** (see next section), then edit the configuration.


In [ ]:
# @@ Install / upgrade required libraries (Colab already has pandas/requests) @@
!pip -q install paramiko python-dateutil tqdm pandas requests

## Configuration

Edit the cell below to set **environment variables** or direct constants that the script relies on.


In [ ]:
# @title Configuration {"run":"auto"}

# @@ Runtime settings (edit freely) @@
import os, json
from dataclasses import asdict

# --- Minimum required ---
ODIN_API_BASE_URL = "" # @param {"type":"string"}
ODIN_API_USERNAME = "" # @param {"type":"string"}
ODIN_API_PASSWORD = "" # @param {"type":"string"}
start_date = "2025-07-01" # @param {"type":"date"}
end_date = "2025-07-01" # @param {"type":"date"}
SFTP_HOST = "" # @param {"type":"string"}
SFTP_USERNAME = "" # @param {"type":"string"}
SFTP_PASSWORD = "" # @param {"type":"string"}
SMTP_HOST = "" # @param {"type":"string"}
SMTP_USERNAME = "" # @param {"type":"string"}
SMTP_PASSWORD = "" # @param {"type":"string"}
SMTP_FROM = "" # @param {"type":"string","placeholder":"report@example.com"}
SMTP_TO = "" # @param {"type":"string","placeholder":"you@example.com"}
SMTP_PORT = "" # @param {"type":"string","placeholder":"587"}
SFTP_REMOTE_PATH = ""  # @param {"type":"string","placeholder":"/reports"}

start_date = start_date + " 00:00:00"
end_date = end_date + " 23:59:59"

# --- Minimum required ---
os.environ['ODIN_API_BASE_URL'] = ODIN_API_BASE_URL
os.environ['ODIN_API_USERNAME'] = ODIN_API_USERNAME
os.environ['ODIN_API_PASSWORD'] = ODIN_API_PASSWORD

# --- Optional overrides ---
os.environ['REPORT_START_DATE'] = start_date
os.environ['REPORT_END_DATE'] = end_date
os.environ['BATCH_SIZE'] = '200'

# SFTP (leave blank to disable)
os.environ['SFTP_HOST'] = SFTP_HOST
os.environ['SFTP_USERNAME'] = SFTP_USERNAME
os.environ['SFTP_PASSWORD'] = SFTP_PASSWORD

# SMTP (leave blank to disable)
os.environ['SMTP_HOST'] = SMTP_HOST
os.environ['SMTP_PORT'] = SMTP_PORT
os.environ['SMTP_USERNAME'] = SMTP_USERNAME
os.environ['SMTP_PASSWORD'] = SMTP_PASSWORD
os.environ['SMTP_FROM'] = SMTP_FROM
os.environ['SMTP_TO'] = SMTP_TO
os.environ['SFTP_REMOTE_PATH'] = SFTP_REMOTE_PATH

print("✅ Configuration Settings Loaded")

## 📥 Load `daily_odin_report.py`

* If running in Colab, upload the script when prompted below (first‑time run)  
* If the script lives on GitHub/GCS, you can `wget`/`gsutil cp` instead.


In [ ]:
# === Fresh-upload & import daily_odin_report.py ============================
import json
from dataclasses import asdict
import os, shutil, importlib.util, sys, re, textwrap
from google.colab import files  # pyright: ignore[reportMissingImports]

# 1️⃣  Make sure any old copy is removed so we *must* upload a new one
for fname in ("daily_odin_report.py",):
    if os.path.exists(fname):
        os.remove(fname)

# 2️⃣  Prompt user to upload a fresh file every run
uploaded = files.upload()                      # ⇧ choose your latest daily_odin_report.py
if not uploaded:
    raise FileNotFoundError("You must upload daily_odin_report.py to continue")

src_name = next(iter(uploaded))                # first (and usually only) uploaded file
shutil.move(src_name, "daily_odin_report.py")  # rename / overwrite

# 3️⃣  Hot-patch the mutable-default list issue, if still present
path = "daily_odin_report.py"
txt  = open(path).read()

if "smtp_to: List[str] =" in txt and "field(" not in txt:       # unpatched version
    # ensure 'field' is imported
    if "from dataclasses import" in txt:
        txt = re.sub(r"from dataclasses import ([^\n]+)",
                     lambda m: ("field" in m.group(1) and m.group(0) or
                                 f"from dataclasses import {m.group(1)}, field"),
                     txt, 1)
    else:
        txt = "from dataclasses import field\n" + txt

    # replace smtp_to line with default_factory version
    txt = re.sub(
        r"smtp_to\s*:\s*List\[str\]\s*=\s*[^\n]+",
        textwrap.dedent("""\
            smtp_to: List[str] = field(
                default_factory=lambda: [
                    a.strip() for a in os.getenv('SMTP_TO', '').split(',') if a.strip()
                ]
            )"""),
        txt, 1
    )
    open(path, "w").write(txt)
    print("🔧  Applied mutable-default patch")

# 4️⃣  Dynamic import
spec = importlib.util.spec_from_file_location("daily_odin_report", path)
dcr  = importlib.util.module_from_spec(spec)
sys.modules["daily_odin_report"] = dcr
spec.loader.exec_module(dcr)

print("✅  daily_odin_report.py uploaded & imported fresh")

## 🔑 Authenticate & Create API Client

In [ ]:
cfg = dcr.Config()
print(json.dumps(asdict(cfg), indent=2))
cfg = dcr.Config()                       # pick up env vars
api_client = dcr.OdinAPIClient(cfg)
assert api_client.authenticate(), "API authentication failed ❌"
print("Authenticated ✔")

## 🏢 Fetch All Service Providers

In [ ]:
# Get all Service Providers
service_providers = api_client.get_service_providers()
len(service_providers), service_providers[:5]

## 🏢 Fetch Service Providers from list

In [ ]:
# Get Service Providers from the list
SERVICE_PROVIDER_LIST = ["ent.odin"] # @param {"type":"string"}

all_sps = api_client.get_service_providers()
service_providers = [
    sp for sp in all_sps
    if sp.get('serviceProviderId') in SERVICE_PROVIDER_LIST
]
len(service_providers), service_providers[0] if service_providers else None

## Fetch Call Records for All Users (batched)

In [ ]:
from tqdm.auto import tqdm

start_date, end_date = dcr.DailyCallReportGenerator(cfg)._get_date_range()
all_call_records = []
total_users = 0
base = cfg.api_base_url.rstrip('/')

for sp in tqdm(service_providers, desc='Service Providers'):
    users    = api_client.get_users_for_service_provider(sp['serviceProviderId'])
    user_ids = [u['userId'] for u in users if u.get('userId')]
    total_users += len(user_ids)

    # 👉 One request per user
    for user_id in tqdm(user_ids, desc='Users', leave=False):
        resp = api_client.session.get(
            f"{base}/api/v2/users/call-records/details",
            params={
                'userIds':   [user_id],
                'startTime': start_date,
                'endTime':   end_date
            }
        )
        resp.raise_for_status()
        jr = resp.json()
        all_call_records.extend(jr.get('data', []))

print(f"Fetched {len(all_call_records):,} records for {total_users:,} users")

## 🗃️ Process & Aggregate Call Records Data

In [ ]:
## 🗃️ Process & Aggregate Call Records Data

# If you have run the "Fetch Call Records for All Users (batched)" cell and have all_call_records:
if 'all_call_records' in globals() and all_call_records:
    print("Processing and aggregating call records data...")
    processor = dcr.CallRecordProcessor(cfg)
    df_raw, df_agg = processor.process_call_records(all_call_records)
    print(f"Processed call records shape: {df_raw.shape}")
    print(f"Aggregated call records shape: {df_agg.shape}")
    if not df_agg.empty:
        print("\nAggregated Call Records by Service Provider:")
        print(df_agg.head())
else:
    print("No call records data found. Please run the call records fetch cell first.")

## 💾 Export Call Records Data to CSV

In [ ]:
# 💾 Export Call Records Data to CSV

# If you have run the "Process & Aggregate Call Records Data" cell and have df_raw and df_agg:
if 'df_raw' in globals() and 'df_agg' in globals():
    exporter = dcr.ReportExporter(cfg)
    raw_csv, agg_csv = exporter.export_to_csv(df_raw, df_agg)
    print('Call Records Files:', raw_csv, agg_csv)
else:
    print("No processed call records data found. Please run the call records processing cell first.")

## ☁️ Optional Outputs (SFTP Upload & Email)

In [ ]:
## ☁️ Optional Outputs (SFTP Upload & Email)

uploaded = False; emailed = False
if cfg.sftp_host:
    uploaded = exporter.upload_to_sftp([raw_csv, agg_csv])
if cfg.smtp_host:
    summary = dcr.DailyCallReportGenerator(cfg)._calculate_summary_stats(
        df_raw, df_agg, len(service_providers), total_users
    )
    emailed = exporter.send_email_report([raw_csv, agg_csv], summary)
print('SFTP:', uploaded, 'Email:', emailed)

---
### ✅ Notebook Complete
Generated 2025-06-27 21:03 UTC